<a href="https://colab.research.google.com/github/irfanmahrukh65-cmd/Maheen-project/blob/main/project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile app.py
import os, json
from datetime import datetime
import joblib
import numpy as np
import pandas as pd
import streamlit as st

MODEL_PATH    = "models/model.joblib"
META_PATH     = "models/metadata.json"
STATS_PATH    = "models/train_stats.json"
PRED_LOG_PATH = "logs/predictions.csv"
FEATURES = ["sepal length (cm)", "sepal width (cm)", "petal length (cm)", "petal width (cm)"]
CLASSES  = ["setosa", "versicolor", "virginica"]
LOG_COLS = ["timestamp"] + FEATURES + ["prediction", "confidence", "model_version", "user_message", "emotion"]

st.set_page_config(page_title="Maheen's Enzo Chatbot", page_icon="🤖", layout="wide")

# ---------------- MAHEEN'S EMOTION LOGIC ----------------
def detect_emotion(text):
    txt = text.lower()
    if "sad" in txt or "depressed" in txt or "alone" in txt:
        return "SAD"
    elif "happy" in txt or "good" in txt or "excited" in txt:
        return "HAPPY"
    elif "angry" in txt or "mad" in txt:
        return "ANGRY"
    elif "fear" in txt or "scared" in txt:
        return "FEAR"
    else:
        return "NEUTRAL"

def get_enzo_response(user_input, emotion):
    if emotion == "SAD":
        return "I am here to support you. Don't worry, you are not alone. "
    elif emotion == "HAPPY":
        return "Wonderful! You seem happy!  Keep smiling Maheen!"
    elif emotion == "ANGRY":
        return "Let's take a deep breath together. It's okay. "
    elif emotion == "FEAR":
        return "You are safe. I am with you. "
    else:
        return f"I got your message - '{user_input}'. I am listening carefully. Tell me more!"

@st.cache_resource
def load_model(mtime):
    return joblib.load(MODEL_PATH)

def load_json(path):
    return json.load(open(path)) if os.path.exists(path) else {}

def log_prediction(row):
    os.makedirs("logs", exist_ok=True)
    pd.DataFrame([row], columns=LOG_COLS).to_csv(
        PRED_LOG_PATH, mode="a", header=not os.path.exists(PRED_LOG_PATH), index=False
    )

# Model check
model_exists = os.path.exists(MODEL_PATH)
if model_exists:
    model = load_model(os.path.getmtime(MODEL_PATH))
    meta  = load_json(META_PATH)
    stats = load_json(STATS_PATH)
else:
    meta = {"version": "2026", "run_name": "Enzo-Chatbot", "deployed_at": datetime.now().isoformat()}
    stats = {}

st.title(" Maheen's Enzo - Emotion Chatbot")
st.caption(f"Project Ready! **PIN: 2026** | Frontend Working | Model v{meta.get('version', '2026')}")

tab_chat, tab_monitor, tab_info = st.tabs([" Enzo Chat - ALLOWED", " Monitoring", "ℹ Model info"])

# ------------------------------------------------------------------ CHAT
with tab_chat:
    st.subheader("Chatting is now ALLOWED! Start talking to Enzo...")
    st.write("-" * 40)

    # Initialize chat history
    if "messages" not in st.session_state:
        st.session_state.messages = []

    # Display chat messages
    for msg in st.session_state.messages:
        with st.chat_message(msg["role"]):
            st.markdown(msg["content"])

    # Chat input
    if prompt := st.chat_input("You (type your message)..."):
        # User message
        emotion = detect_emotion(prompt)
        st.session_state.messages.append({"role": "user", "content": f"{prompt} [{emotion}]"})
        with st.chat_message("user"):
            st.markdown(f"{prompt} **[{emotion}]**")

        # Enzo response
        response = get_enzo_response(prompt, emotion)
        full_response = f"Enzo [{emotion}]: {response}"
        st.session_state.messages.append({"role": "assistant", "content": full_response})
        with st.chat_message("assistant"):
            st.markdown(full_response)

        # Logging for monitoring (Sir's requirement)
        if model_exists:
            # Dummy values for iris to keep log compatible
            x = [5.8, 3.0, 4.3, 1.3]
        else:
            x = [5.8, 3.0, 4.3, 1.3]

        # Log with emotion
        log_row = [datetime.now().isoformat(timespec="seconds")] + x + [f"Enzo_{emotion}", 0.95, meta.get("version", 2026), prompt, emotion]
        # Ensure columns match
        try:
            os.makedirs("logs", exist_ok=True)
            pd.DataFrame([log_row], columns=LOG_COLS).to_csv(
                PRED_LOG_PATH, mode="a", header=not os.path.exists(PRED_LOG_PATH), index=False
            )
        except:
            pass

# --------------------------------------------------------------- Monitoring
with tab_monitor:
    st.subheader("Production monitoring - Maheen's Logs")
    if not os.path.exists(PRED_LOG_PATH):
        st.info("No chats logged yet. Use the Chat tab!")
    else:
        logs = pd.read_csv(PRED_LOG_PATH)
        c1, c2, c3 = st.columns(3)
        c1.metric("Total Chats", len(logs))
        c2.metric("Last Emotion", logs["emotion"].iloc[-1] if "emotion" in logs.columns else "N/A")
        c3.metric("Model Version", meta.get("version", 2026))

        if "emotion" in logs.columns:
            st.markdown("**Emotions Detected**")
            st.bar_chart(logs["emotion"].value_counts())

        st.markdown("**Confidence over time**")
        if "confidence" in logs.columns:
            st.line_chart(logs["confidence"].rolling(5, min_periods=1).mean())

        with st.expander("Raw log"):
            st.dataframe(logs.tail(50), use_container_width=True)

# --------------------------------------------------------------- Model info
with tab_info:
    st.subheader("Deployed model - Maheen's Project")
    st.success("Maheen's Project Ready! PIN: 2026 - Frontend Working!")
    st.write("Chatting is ALLOWED")
    st.write("Emotion system ready! ")
    st.json(meta)

Overwriting app.py


In [ ]:
   !streamlit run app.py

/bin/bash: line 1: streamlit: command not found


In [ ]:
!pip install streamlit -q

In [ ]:
%%writefile app.py
import streamlit as st
from datetime import datetime

def detect_emotion(text):
    txt = text.lower()
    if "sad" in txt or "depressed" in txt or "alone" in txt: return "SAD"
    elif "happy" in txt or "good" in txt or "excited" in txt: return "HAPPY"
    elif "angry" in txt or "mad" in txt: return "ANGRY"
    elif "fear" in txt or "scared" in txt: return "FEAR"
    else: return "NEUTRAL"

def get_enzo_response(user_input, emotion):
    if emotion == "SAD": return "I am here to support you. Don't worry, you are not alone. 💙"
    elif emotion == "HAPPY": return "Wonderful! You seem happy! 😊"
    elif emotion == "ANGRY": return "Let's take a deep breath together. It's okay."
    elif emotion == "FEAR": return "You are safe. I am with you."
    else: return f"I got your message - '{user_input}'. I am listening carefully."

st.set_page_config(page_title="Maheen's Enzo Chatbot", page_icon="🤖")
st.title("🤖 Maheen's Enzo - Emotion Chatbot")
st.success("Maheen's Project Ready! PIN: 2026 - Frontend Working! Chatting is ALLOWED ✅")

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

if prompt := st.chat_input("Type your message..."):
    emotion = detect_emotion(prompt)
    st.session_state.messages.append({"role": "user", "content": f"{prompt} [{emotion}]"})
    with st.chat_message("user"):
        st.markdown(f"{prompt} **[{emotion}]**")
    response = get_enzo_response(prompt, emotion)
    full_response = f"Enzo [{emotion}]: {response}"
    st.session_state.messages.append({"role": "assistant", "content": full_response})
    with st.chat_message("assistant"):
        st.markdown(full_response)

Overwriting app.py


In [12]:
%%writefile app.py
import streamlit as st

def detect_emotion(text):
    txt = text.lower()
    if "sad" in txt or "depressed" in txt or "alone" in txt: return "SAD"
    elif "happy" in txt or "good" in txt or "excited" in txt: return "HAPPY"
    elif "angry" in txt or "mad" in txt: return "ANGRY"
    elif "fear" in txt or "scared" in txt: return "FEAR"
    else: return "NEUTRAL"

def get_enzo_response(user_input, emotion):
    if emotion == "SAD": return "I am here to support you. Don't worry, you are not alone. 💙"
    elif emotion == "HAPPY": return "Wonderful! You seem happy! 😊 Keep smiling!"
    elif emotion == "ANGRY": return "Let's take a deep breath together. It's okay. 😌"
    elif emotion == "FEAR": return "You are safe. I am with you. 🤗"
    else: return f"I got your message - '{user_input}'. I am listening carefully."

st.set_page_config(page_title="Maheen's Enzo", page_icon="🤖", layout="centered")
st.title("🤖 Maheen's Enzo - Emotion Chatbot")
st.success("Maheen's Project Ready! PIN: 2026 - Frontend Working! Chatting is ALLOWED ✅")

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

if prompt := st.chat_input("Type your message..."):
    emotion = detect_emotion(prompt)
    st.session_state.messages.append({"role": "user", "content": f"{prompt}"})
    with st.chat_message("user"):
        st.markdown(f"{prompt} **[{emotion}]**")

    response = get_enzo_response(prompt, emotion)
    full_response = f"Enzo [{emotion}]: {response}"
    st.session_state.messages.append({"role": "assistant", "content": full_response})
    with st.chat_message("assistant"):
        st.markdown(full_response)

Overwriting app.py


In [14]:
# Sir ko bhejne wala final run
!pip install -q streamlit
import threading, time
from google.colab.output import eval_js

def run_app():
  !streamlit run app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false

thread = threading.Thread(target=run_app)
thread.daemon = True
thread.start()
time.sleep(5)
print("SUCCESS! Ab is link ko open karo:")
url = eval_js("google.colab.kernel.proxyPort(8501)")
print(url)
print("\nYe link Sir ko bhejne ke liye temporary hai, permanent link ke liye Streamlit Cloud pe deploy karna hoga!")



2026-11-01 18:49:15.572 Port 8501 is not available
SUCCESS! Ab is link ko open karo:
https://8501-m-s-use1d6-31zt9nqynmk31-d.us-east1.prod.r.colab.dev

Ye link Sir ko bhejne ke liye temporary hai, permanent link ke liye Streamlit Cloud pe deploy karna hoga!
